# 4×L4 Scorer-Hardware Benchmark

Measures the three numbers needed to set `max_time_minutes` safely for the Gemma 4 SWE agent competition:

1. **vLLM server startup time** (one-time cost subtracted from the 12h global ceiling)
2. **Per-task Container A setup time** (snapshot extract + pip install + wheel stream + baseline commit)
3. **Single-stream decode speed** (tokens/s on this hardware with realistic prompts)
4. **Concurrency scaling** (per-stream retained speed at 2/3/4 concurrent requests — de-risks the parallel-solver design)

Then computes the safe `max_time_minutes` from the formula:

```
safe_agent_s = (43200 − startup_s − N_tasks × setup_s) / N_tasks
safe_max_time_minutes = safe_agent_s / 60 × (1 − margin)
```

---
**Prerequisites — read before running:**
- **Accelerator:** This notebook requires 4×L4 GPUs (the exact scorer hardware). On Kaggle, verify your accelerator type. If 4×L4 is not available as a notebook option, use a GCP `g2-standard-48` instance with 4×NVIDIA L4 GPUs instead.
- **Model weights:** Attach the competition model (`gemma-4-31b-it-qat-w4a16-ct`) as a dataset/model input and set `MODEL_PATH` in the config cell below. The model likely appears under `/kaggle/input/` after you attach it via 'Add data'.
- **Competition data:** Attach the competition data (`swegemma` tasks + snapshots). The tasks.jsonl and snapshots dir are needed for the setup-time measurement.
- **Internet on:** Enable internet access in the notebook settings (needed to pip-install packages).
- **This benchmark does NOT count as a submission.** It is a plain notebook, not a `submission.zip` upload.


In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────
# Adjust these paths after attaching the model and competition data.

import os, glob

# Model weights (INT4 quantized, compressed-tensors format)
# Common Kaggle mount point: /kaggle/input/<model-slug>/
# Find it with: glob.glob('/kaggle/input/*/*')[:5]
MODEL_PATH = os.environ.get('MODEL_PATH', '')
if not MODEL_PATH:
    candidates = glob.glob('/kaggle/input/**/config.json', recursive=True)
    if candidates:
        MODEL_PATH = os.path.dirname(candidates[0])
        print(f'Auto-detected model at: {MODEL_PATH}')
    else:
        MODEL_PATH = '/kaggle/input/gemma-4-31b-it-qat-w4a16-ct'  # fallback guess
        print(f'WARNING: could not auto-detect model — using default {MODEL_PATH}')
print('MODEL_PATH:', MODEL_PATH)

# Tasks + snapshots (competition data)
TASKS_JSONL      = '/kaggle/input/gemma-4-developer-agent/tasks.jsonl'
SNAPSHOTS_DIR    = '/kaggle/input/gemma-4-developer-agent/snapshots'
if not os.path.exists(TASKS_JSONL):
    # Try local fallback (if running in the repo, not on Kaggle)
    TASKS_JSONL   = 'competition_data/tasks.jsonl'
    SNAPSHOTS_DIR = 'competition_data/snapshots'
print('TASKS_JSONL:', TASKS_JSONL, '| exists:', os.path.exists(TASKS_JSONL))

# Budget constants
GLOBAL_BUDGET_S  = 43200   # 12 h in seconds
N_TASKS          = 120     # confirmed from competition Data tab
SAFETY_MARGIN    = 0.10    # 10 % buffer

# vLLM port (scorer uses a single TP=4 server on port 8000)
VLLM_PORT = 8000
VLLM_URL  = f'http://127.0.0.1:{VLLM_PORT}/v1'

In [ ]:
# ── INSTALL ────────────────────────────────────────────────────────────────────
# Install vLLM and the swegemma harness.  Use pinned versions.
# vLLM: pin to the version the competition image uses (check HARNESS_README for 'vllm==');
# if unspecified, a recent release that supports Gemma 4 / compressed-tensors is fine.
import subprocess, sys

def run(cmd, **kw):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=True, **kw)

run('pip install -q vllm>=0.9 compressed-tensors openai')

# Install swegemma harness wheels (provided in the competition data)
# Try local wheelhouse first, fall back to PyPI if not present.
whl_dir = '/kaggle/input/gemma-4-developer-agent/wheels'  # adjust if needed
if os.path.isdir(whl_dir):
    run(f'pip install -q --no-index --find-links={whl_dir} swegemma adk-submission adk-eval-core google-adk')
else:
    print('Wheels dir not found; attempting PyPI install (may not match scorer version)')
    run('pip install -q swegemma adk-submission adk-eval-core google-adk')

print('Installs done')

In [ ]:
# ── SECTION 1: vLLM SERVER STARTUP TIME ───────────────────────────────────────
# Launch with the EXACT scorer configuration from HARNESS_README §3.1:
#   tensor_parallel_size=4, gpu_memory_utilization=0.90, max_model_len=32768,
#   enable_lora, max_loras=8, max_lora_rank=128, tool_call_parser=gemma4,
#   reasoning_parser=gemma4.
# NOTE: one TP=4 server serving a single endpoint — matches the scorer topology.

import time, threading, requests, subprocess, json

SERVED_MODEL = 'gemma-4-31b-it-qat-w4a16-ct'

vllm_cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_PATH,
    '--served-model-name', SERVED_MODEL,
    '--host', '127.0.0.1',
    '--port', str(VLLM_PORT),
    '--tensor-parallel-size', '4',
    '--gpu-memory-utilization', '0.90',
    '--max-model-len', '32768',
    '--enable-lora',
    '--max-loras', '8',
    '--max-lora-rank', '128',
    '--enable-auto-tool-choice',
    '--tool-call-parser', 'gemma4',
    '--reasoning-parser', 'gemma4',
    '--reasoning-config', json.dumps({
        'reasoning_start_str': '<|channel>',
        'reasoning_end_str': '<channel|>'
    }),
    '--default-chat-template-kwargs', json.dumps({'enable_thinking': True}),
    '--no-scheduler-reserve-full-isl',
    '--language-model-only',
    '--skip-mm-profiling',
    '--max-num-seqs', '8',
    '--max-num-batched-tokens', '8192',
    '--cudagraph-capture-sizes', '1', '2', '4', '8',
]

print('Launching vLLM server...')
t0_server = time.time()
vllm_proc = subprocess.Popen(
    vllm_cmd,
    stdout=open('/kaggle/working/vllm.log', 'w'),
    stderr=subprocess.STDOUT,
    env={**os.environ, 'VLLM_NO_USAGE_STATS': '1'},
)

# Poll until /v1/models responds
while True:
    try:
        r = requests.get(f'{VLLM_URL}/models', timeout=5)
        if r.status_code == 200:
            break
    except Exception:
        pass
    if vllm_proc.poll() is not None:
        raise RuntimeError('vLLM process exited unexpectedly — check /kaggle/working/vllm.log')
    elapsed = time.time() - t0_server
    print(f'  [{elapsed:.0f}s] waiting...', end='\r')
    time.sleep(5)

STARTUP_S = time.time() - t0_server
print(f'\nvLLM server ready.  Startup time: {STARTUP_S:.1f} s  ({STARTUP_S/60:.2f} min)')

In [ ]:
# ── SECTION 2: SINGLE-STREAM DECODE SPEED ─────────────────────────────────────
# Build a realistic prompt (~10K token input) matching the T3_7m profile:
#   185K prompt tokens / 17 calls ≈ 10.9K tokens per call.
# Use the actual system prompt + a sample problem_statement from tasks.jsonl,
# padded with the repo instruction preamble to reach target length.

from openai import OpenAI
import numpy as np

client = OpenAI(base_url=f'{VLLM_URL}', api_key='unused')

# Load a real problem statement
with open(TASKS_JSONL) as f:
    sample_task = json.loads(f.readline())
problem_statement = sample_task['problem_statement']
repo = sample_task['repo']

# Read the actual system prompt (T3_7m tree)
SYSTEM_MD = 'submission_build/_trees/t3_7m/prompts/system.md'
if os.path.exists(SYSTEM_MD):
    with open(SYSTEM_MD) as f:
        system_prompt = f.read()
else:
    system_prompt = 'You are an autonomous software engineer. Fix the bug described in the problem statement.'

user_msg = (
    f'You are evaluating a software engineering task for repository {repo}.\n\n'
    f'Problem Statement:\n{problem_statement}\n\n'
    f'## Task Budget\n- Time allowance: 7.0 minutes\n- Tool calls: 28\n'
)

# Approximate additional context (tool results from earlier calls) to reach ~10K tokens
# by repeating a realistic tool-result snippet
tool_context_chunk = (
    '<tool_call_result>{"status": "ok", "filepath": "src/module.py", "content": "'
    + 'class Handler:\n    def process(self, data):\n        return data\n' * 80
    + '"}\n</tool_call_result>\n'
)
# Build conversation history to ~10K prompt tokens (rough approximation: 1 token ≈ 4 chars)
target_chars = 10_000 * 4
existing_chars = len(system_prompt) + len(user_msg)
repeats = max(1, (target_chars - existing_chars) // len(tool_context_chunk))
padded_user = user_msg + tool_context_chunk * repeats
print(f'Prompt size: ~{(len(system_prompt) + len(padded_user)) // 4:,} tokens (estimated)')

messages = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'user', 'content': padded_user},
]

def measure_decode(n_tokens=256, n_trials=5):
    """Measure single-stream decode speed: mean tokens/s and TTFT over n_trials runs."""
    speeds, ttfts = [], []
    for trial in range(n_trials):
        t0 = time.time()
        first_token_time = None
        token_count = 0
        stream = client.chat.completions.create(
            model=SERVED_MODEL,
            messages=messages,
            max_tokens=n_tokens,
            temperature=0.2,
            stream=True,
            extra_body={'thinking_budget': 1024, 'include_thoughts': False},
        )
        for chunk in stream:
            delta = chunk.choices[0].delta.content if chunk.choices else ''
            if delta:
                if first_token_time is None:
                    first_token_time = time.time() - t0
                token_count += 1  # rough, streaming gives one token per chunk in vLLM
        elapsed = time.time() - t0
        if token_count > 0 and elapsed > 0:
            speeds.append(token_count / elapsed)
            ttfts.append(first_token_time or elapsed)
        print(f'  trial {trial+1}: {token_count} tokens in {elapsed:.1f}s '
              f'({token_count/elapsed:.1f} tok/s, TTFT {(first_token_time or elapsed)*1000:.0f}ms)')
    return speeds, ttfts

print('Measuring single-stream decode (5 trials × 256 completion tokens)...')
speeds, ttfts = measure_decode(n_tokens=256, n_trials=5)
DECODE_TOKS_PER_S = float(np.median(speeds))
print(f'\nSingle-stream decode: {DECODE_TOKS_PER_S:.1f} tok/s (median)')
print(f'TTFT: {np.median(ttfts)*1000:.0f} ms (median)')
print(f'At {DECODE_TOKS_PER_S:.0f} tok/s: 0.038 s/call would need {0.038*DECODE_TOKS_PER_S:.0f} tokens/call')

In [ ]:
# ── SECTION 3: CONCURRENCY SCALING ────────────────────────────────────────────
# Fire N concurrent requests to the same TP=4 server and measure per-stream
# retained decode speed.  This de-risks the parallel-solver (Design A) idea.
# Gate: per-stream speed at N=3 should be ≥ 75 % of N=1 to proceed with Design A.

import concurrent.futures

def single_request(n_tokens=128):
    t0 = time.time()
    token_count = 0
    stream = client.chat.completions.create(
        model=SERVED_MODEL,
        messages=messages,
        max_tokens=n_tokens,
        temperature=0.2,
        stream=True,
        extra_body={'thinking_budget': 1024, 'include_thoughts': False},
    )
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            token_count += 1
    elapsed = time.time() - t0
    return token_count, elapsed

def measure_concurrency(n, n_tokens=128):
    t0 = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=n) as pool:
        results = list(pool.map(lambda _: single_request(n_tokens), range(n)))
    wall = time.time() - t0
    per_stream_tps = [r[0]/r[1] for r in results if r[1] > 0]
    return per_stream_tps, wall

print('Measuring concurrency scaling (1 warm-up, then N=1,2,3,4)...')
# Warm-up
single_request(32)

baseline_tps = None
concurrency_results = {}
for n in [1, 2, 3, 4]:
    tps_list, wall = measure_concurrency(n)
    mean_tps = float(np.mean(tps_list))
    concurrency_results[n] = {'per_stream_tps': mean_tps, 'wall_s': wall}
    if n == 1:
        baseline_tps = mean_tps
    retained = mean_tps / baseline_tps if baseline_tps else 0
    gate = '✅ PASS' if retained >= 0.75 else '❌ FAIL'
    print(f'  N={n}: {mean_tps:.1f} tok/s per stream  {retained*100:.0f}% of N=1  {gate} (gate ≥75%)')

print('\nConclusion for Design A (parallel best-of-N solvers):')
if concurrency_results.get(3, {}).get('per_stream_tps', 0) / baseline_tps >= 0.75:
    print('  Concurrency 3 passes the gate. Parallel solvers are viable on this hardware.')
else:
    print('  Concurrency 3 fails the gate. Use P-lite (read-only parallel analyzers + single solver).')

In [ ]:
# ── SECTION 4: PER-TASK CONTAINER A SETUP TIME ────────────────────────────────
# Strategy:
#   Run A: swegemma eval --task-id X (full: setup_A + agent + setup_B + verify)
#           with max_time_minutes=0.05 so the agent loop ends in ~3 seconds
#           → task_results.jsonl gives agent_only_s = duration_seconds
#   Run B: swegemma eval --task-id X --skip-agent-patch (phase 2 only: setup_B + verify)
#           → total wall time ≈ setup_B + verify_B
#   setup_A ≈ wall_A - agent_only_s - wall_B   (setup_B ≈ setup_A for same repo)
# Repeat for 3 different tasks (fastapi, rich, requests if available) for variance.
# Use --sandbox subprocess since Kaggle notebooks don't support Docker-in-Docker.

import shutil, json, subprocess, time, os, glob, pathlib

SAMPLE_TASK_IDS = []  # will auto-pick below
with open(TASKS_JSONL) as f:
    all_tasks = [json.loads(l) for l in f]
# Pick one task from each available repo
seen_repos = set()
for t in all_tasks:
    if t['repo'] not in seen_repos and len(SAMPLE_TASK_IDS) < 4:
        SAMPLE_TASK_IDS.append(t['instance_id'])
        seen_repos.add(t['repo'])
print('Benchmark task IDs:', SAMPLE_TASK_IDS)

BASE_RESULTS = '/kaggle/working/bench_results'

def swegemma_eval(task_id, results_subdir, extra_flags=''):
    results_dir = f'{BASE_RESULTS}/{results_subdir}'
    os.makedirs(results_dir, exist_ok=True)
    # Build the swegemma eval command
    # Using a minimal submission dir pointing at our T3_7m arm (or sample_submission)
    submission_dir = 'submission_build/_trees/t3_7m'
    if not os.path.isdir(submission_dir):
        submission_dir = 'kaggle_provided/sample_submission'
    cmd = (
        f'swegemma eval '
        f'--tasks {TASKS_JSONL} '
        f'--snapshots-dir {SNAPSHOTS_DIR} '
        f'--submission-dir {submission_dir} '
        f'--results-dir {results_dir} '
        f'--task-id {task_id} '
        f'--sandbox subprocess '
        f'--max-tool-calls 1 '
        f'--max-time-minutes 0.05 '
        f'--max-turns 2 '
        f'{extra_flags}'
    )
    t0 = time.time()
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    wall = time.time() - t0
    # Parse agent duration from task_results.jsonl
    agent_s = None
    tr_path = f'{results_dir}/task_results.jsonl'
    if os.path.exists(tr_path):
        with open(tr_path) as f:
            for line in f:
                rec = json.loads(line)
                if rec.get('instance_id') == task_id:
                    agent_s = rec.get('duration_seconds')
                    break
    return wall, agent_s, result.returncode

setup_times = []
print('Measuring per-task Container A setup time (subprocess backend)...')
for task_id in SAMPLE_TASK_IDS:
    print(f'\n  Task: {task_id}')
    # Run A: full run (setup_A + agent)
    wall_A, agent_s, rc_A = swegemma_eval(task_id, f'{task_id}_full')
    # Run B: skip-agent-patch (setup_B + verify only, no agent loop)
    wall_B, _, rc_B = swegemma_eval(task_id, f'{task_id}_skip', '--skip-agent-patch')
    print(f'    wall_A={wall_A:.1f}s  agent_only={agent_s:.1f}s  wall_B={wall_B:.1f}s  rc={rc_A},{rc_B}')
    if agent_s is not None:
        setup_A = wall_A - (agent_s or 0) - wall_B
        setup_times.append(max(0, setup_A))  # floor at 0 for numerical noise
        print(f'    setup_A ≈ {setup_A:.1f}s')
    else:
        print(f'    WARNING: could not parse agent duration from task_results.jsonl')

if setup_times:
    SETUP_A_S = float(np.median(setup_times))
    print(f'\nPer-task setup time (median): {SETUP_A_S:.1f} s  range [{min(setup_times):.1f}, {max(setup_times):.1f}]')
else:
    SETUP_A_S = 30.0  # conservative fallback if measurement failed
    print(f'WARNING: setup measurement failed, using fallback {SETUP_A_S}s')

In [ ]:
# ── SECTION 5: COMPUTE SAFE max_time_minutes ──────────────────────────────────

safe_agent_s = (GLOBAL_BUDGET_S - STARTUP_S - N_TASKS * SETUP_A_S) / N_TASKS
safe_agent_s_with_margin = safe_agent_s * (1 - SAFETY_MARGIN)
safe_min = safe_agent_s_with_margin / 60

print('=' * 60)
print('BENCHMARK SUMMARY')
print('=' * 60)
print(f'  Global budget:          {GLOBAL_BUDGET_S} s (12 h, incl. setup)')
print(f'  Hidden task count:      {N_TASKS}')
print(f'  vLLM server startup:    {STARTUP_S:.1f} s  ({STARTUP_S/60:.2f} min)')
print(f'  Per-task setup (A):     {SETUP_A_S:.1f} s  ({SETUP_A_S*N_TASKS:.0f} s total)')
print(f'  Remaining for agent:    {safe_agent_s:.1f} s / task  ({safe_agent_s/60:.2f} min)')
print(f'  After {SAFETY_MARGIN*100:.0f}% margin:        {safe_agent_s_with_margin:.1f} s  → {safe_min:.2f} min')
print()
print(f'  ✅ Recommended max_time_minutes: {safe_min:.1f}')
print()
print(f'  Decode speed (single-stream): {DECODE_TOKS_PER_S:.1f} tok/s')
print(f'    → {safe_agent_s_with_margin:.0f}s × {DECODE_TOKS_PER_S:.0f} tok/s = '
      f'{safe_agent_s_with_margin*DECODE_TOKS_PER_S:,.0f} completion tokens per task budget')
print(f'    → at 17 calls/task: ~{safe_agent_s_with_margin*DECODE_TOKS_PER_S/17:.0f} tokens/call on average')
print()
print('Concurrency gate (Design A — parallel best-of-N solvers):')
for n, r in concurrency_results.items():
    retained = r['per_stream_tps'] / baseline_tps if baseline_tps else 0
    gate = '✅' if retained >= 0.75 else '❌'
    print(f'  N={n}: {retained*100:.0f}% retained  {gate}')

print()
print('Comparison to current eval_config.yaml:')
for cur_min, label in [(7, 'T3_7m'), (8, 'T1/T3_dynamic'), (6, 'T1_6m')]:
    total_agent_h = (cur_min * 60 * N_TASKS) / 3600
    total_with_setup_h = (cur_min * 60 * N_TASKS + STARTUP_S + N_TASKS * SETUP_A_S) / 3600
    status = '✅ safe' if total_with_setup_h <= 12 else f'❌ {total_with_setup_h:.1f}h > 12h'
    print(f'  {label} ({cur_min}min cap): agent {total_agent_h:.1f}h + overhead → {total_with_setup_h:.1f}h  {status}')

In [ ]:
# ── SECTION 6: SAVE RESULTS ───────────────────────────────────────────────────

results = {
    'startup_s': STARTUP_S,
    'setup_A_s': SETUP_A_S,
    'setup_A_s_all': setup_times,
    'decode_toks_per_s': DECODE_TOKS_PER_S,
    'concurrency_results': concurrency_results,
    'safe_max_time_minutes': safe_min,
    'safe_agent_s': safe_agent_s_with_margin,
    'global_budget_s': GLOBAL_BUDGET_S,
    'n_tasks': N_TASKS,
    'safety_margin': SAFETY_MARGIN,
}

out_path = '/kaggle/working/l4_benchmark_results.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {out_path}')
print(json.dumps(results, indent=2))